# Notebook 04 — Privacy Filter (Face + Text Detection)

Pretrained models — **no training needed**. Sets up, tests, and exports to ONNX for Jetson.

## Cell 04-01 | Mount & Install

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys, subprocess, torch, cv2
import numpy as np
from PIL import Image

BASE    = '/content/drive/MyDrive/ARGUS'
MODELS  = f'{BASE}/models'
EXPORTS = f'{BASE}/exports'
DEVICE  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'insightface', 'onnxruntime-gpu', 'opencv-python-headless'])
print(f'Ready | Device: {DEVICE}')

## Cell 04-02 | Download SCRFD Face Detector

In [ ]:
from huggingface_hub import hf_hub_download
import shutil

PRIVACY_DIR = f'{MODELS}/privacy'
DONE_FLAG   = f'{PRIVACY_DIR}/scrfd_done.flag'
os.makedirs(PRIVACY_DIR, exist_ok=True)

if not os.path.exists(DONE_FLAG):
    url = 'https://github.com/deepinsight/insightface/releases/download/v0.7/buffalo_s.zip'
    out = f'{PRIVACY_DIR}/buffalo_s.zip'
    subprocess.run(['wget', '-q', '--show-progress', '-O', out, url])
    subprocess.run(['unzip', '-q', out, '-d', PRIVACY_DIR])
    open(DONE_FLAG, 'w').close()
    print('SCRFD downloaded')
else:
    print('SCRFD already downloaded')

import insightface
from insightface.app import FaceAnalysis
face_app = FaceAnalysis(name='buffalo_s', root=PRIVACY_DIR,
                        providers=['CUDAExecutionProvider', 'CPUExecutionProvider'])
face_app.prepare(ctx_id=0, det_size=(640, 640))
print('Face detector loaded and ready')

## Cell 04-03 | Test Face Detection & Blur

In [ ]:
import cv2, numpy as np
from PIL import Image
import matplotlib.pyplot as plt

def blur_faces(image_bgr, face_app, blur_strength=51):
    # Detect faces and apply Gaussian blur for privacy protection
    img_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    faces   = face_app.get(img_rgb)
    result  = image_bgr.copy()
    n_faces = len(faces)
    for face in faces:
        bbox = face.bbox.astype(int)
        x1, y1, x2, y2 = bbox[0], bbox[1], bbox[2], bbox[3]
        pad = 20
        x1  = max(0, x1 - pad); y1 = max(0, y1 - pad)
        x2  = min(image_bgr.shape[1], x2 + pad); y2 = min(image_bgr.shape[0], y2 + pad)
        roi = result[y1:y2, x1:x2]
        if roi.size > 0:
            result[y1:y2, x1:x2] = cv2.GaussianBlur(roi, (blur_strength, blur_strength), 0)
    return result, n_faces

test_path = '/tmp/test_image.jpg'
subprocess.run(['wget', '-q', '-O', test_path,
    'https://upload.wikimedia.org/wikipedia/commons/thumb/1/14/Gatto_europeo4.jpg/640px-Gatto_europeo4.jpg'])
img = cv2.imread(test_path)
blurred, n = blur_faces(img, face_app)
print(f'Detected {n} face(s)')
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(cv2.cvtColor(img,     cv2.COLOR_BGR2RGB)); axes[0].set_title('Original')
axes[1].imshow(cv2.cvtColor(blurred, cv2.COLOR_BGR2RGB)); axes[1].set_title('Privacy Protected')
plt.tight_layout()
plt.savefig(f'{BASE}/logs/privacy_test.png')
plt.show()
print('Privacy filter working')

## Cell 04-04 | CRAFT Text Region Detection

In [ ]:
CRAFT_DIR = f'{BASE}/craft'

if not os.path.exists(CRAFT_DIR):
    subprocess.run(['git', 'clone', 'https://github.com/clovaai/CRAFT-pytorch', CRAFT_DIR])
    print('CRAFT cloned')
else:
    print('CRAFT already cloned')

craft_weights = f'{MODELS}/privacy/craft_mlt_25k.pth'
if not os.path.exists(craft_weights):
    url = 'https://github.com/clovaai/CRAFT-pytorch/releases/download/v0.0.0/craft_mlt_25k.pth'
    subprocess.run(['wget', '-q', '--show-progress', '-O', craft_weights, url])
    print('CRAFT weights downloaded')
else:
    print('CRAFT weights already present')

print('CRAFT usage in ARGUS pipeline:')
print('  -> Detects text regions in wide camera feed')
print('  -> Flags regions as potentially_private=True')
print('  -> LLM skips private regions unless user asks explicitly')
print('  -> Face blur runs always; text flagging is advisory')